# Optimize Weld's Experiment with Ax


In [1]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig


In [16]:
# Bootstrap MSVC toolchain inside the kernel process so BoTorch can build fused kernels on Windows.

def _load_msvc_env_from_vcvars() -> bool:
    vcvars = Path(
        r"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat"
    )
    if not vcvars.exists():
        print(f"MSVC bootstrap skipped: {vcvars} not found.")
        return False

    dump_env_cmd = Path(tempfile.gettempdir()) / "_dump_msvc_env.cmd"
    dump_env_cmd.write_text(
        "@echo off\n"
        f"call \"{vcvars}\" >nul\n"
        "if errorlevel 1 exit /b 1\n"
        "set\n",
        encoding="utf-8",
    )

    try:
        proc = subprocess.run(
            ["cmd.exe", "/d", "/c", str(dump_env_cmd)],
            capture_output=True,
            text=True,
            encoding="mbcs",
            errors="replace",
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        print(f"MSVC bootstrap failed with return code {exc.returncode}.")
        if exc.stderr:
            print(exc.stderr.strip())
        return False

    for line in proc.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value

    # Ensure the kernel keeps the active venv Scripts folder on PATH.
    scripts_dir = str(Path(sys.executable).resolve().parent)
    path_parts = os.environ.get("PATH", "").split(os.pathsep)
    if scripts_dir not in path_parts:
        os.environ["PATH"] = scripts_dir + os.pathsep + os.environ.get("PATH", "")

    return True


loaded = _load_msvc_env_from_vcvars()
print("MSVC env loaded:", loaded)
print("cl:", shutil.which("cl"))
print("ninja:", shutil.which("ninja"))

MSVC env loaded: True
cl: C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Tools\MSVC\14.44.35207\bin\HostX64\x64\cl.EXE
ninja: c:\Users\admin\Coding\research\weld-ml-v2\.venv\Scripts\ninja.EXE


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig


In [18]:
NUM_NEW_EXP = 10

In [19]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [20]:
folder_path = Path(os.getcwd()) / "data"

In [21]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,1,205.512265,204.738760,207.056342,183.713911,181.792203,184.483996
1,1,1462,1811,1851,93,63,41,17,1,199.002716,200.430817,198.957274,170.864008,177.489548,176.112604
2,2,1821,1598,1421,62,74,99,20,1,210.138445,208.003040,208.332095,180.967275,193.690799,191.724174
3,3,1894,1873,1756,73,98,82,14,1,218.672547,204.882794,213.804086,189.691174,190.036535,190.747691
4,4,1691,1519,1622,51,41,57,29,1,202.873540,204.304761,209.820085,185.025997,178.124362,183.032944
5,5,1561,1943,1492,42,48,47,12,1,198.329154,196.761871,199.989863,179.458487,181.605291,178.526271
6,6,1957,1466,1959,83,90,92,26,1,218.466581,220.636707,207.647358,193.740708,197.195762,198.211029
7,7,1730,1704,1664,64,83,75,19,0,206.182400,207.074947,205.356296,181.909813,183.217576,181.606649
8,8,1489,1687,1835,92,57,66,23,1,193.481722,194.686593,194.613201,174.629257,168.746364,175.117417
9,9,1549,1920,1641,78,74,59,11,0,197.976671,201.113731,195.416145,179.738015,181.200059,176.411867


In [22]:
print(exp.columns)
cols_advancing = [
    "sigma_advancing_1",
    "sigma_advancing_2",
    "sigma_advancing_3",
]
cols_retreating = [
    "sigma_retreating_1",
    "sigma_retreating_2",
    "sigma_retreating_3",
]

Index(['trial_index', 'R1', 'R2', 'R3', 'W1', 'W2', 'W3', 'D',
       'visual_quality', 'sigma_advancing_1', 'sigma_advancing_2',
       'sigma_advancing_3', 'sigma_retreating_1', 'sigma_retreating_2',
       'sigma_retreating_3'],
      dtype='str')


In [23]:
filt_advancing = exp[cols_advancing].isna().all(axis=1)
filt_retreating = exp[cols_retreating].isna().all(axis=1)
filt_removed = filt_advancing | filt_retreating
filt_removed

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
dtype: bool

In [24]:
exp_todo = exp[filt_removed].copy()
exp = exp[~filt_removed].copy()
display(exp)
display(exp_todo)

,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,1,205.512265,204.738760,207.056342,183.713911,181.792203,184.483996
1,1,1462,1811,1851,93,63,41,17,1,199.002716,200.430817,198.957274,170.864008,177.489548,176.112604
2,2,1821,1598,1421,62,74,99,20,1,210.138445,208.003040,208.332095,180.967275,193.690799,191.724174
3,3,1894,1873,1756,73,98,82,14,1,218.672547,204.882794,213.804086,189.691174,190.036535,190.747691
4,4,1691,1519,1622,51,41,57,29,1,202.873540,204.304761,209.820085,185.025997,178.124362,183.032944
5,5,1561,1943,1492,42,48,47,12,1,198.329154,196.761871,199.989863,179.458487,181.605291,178.526271
6,6,1957,1466,1959,83,90,92,26,1,218.466581,220.636707,207.647358,193.740708,197.195762,198.211029
7,7,1730,1704,1664,64,83,75,19,0,206.182400,207.074947,205.356296,181.909813,183.217576,181.606649
8,8,1489,1687,1835,92,57,66,23,1,193.481722,194.686593,194.613201,174.629257,168.746364,175.117417
9,9,1549,1920,1641,78,74,59,11,0,197.976671,201.113731,195.416145,179.738015,181.200059,176.411867


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3


In [25]:
# 2. Configure where Ax will search.

bound_R = [1400, 2000]  # Rotatinal speed
bound_W = [40, 100]  # Welding speed
bound_D = [10, 30]  # Dwell time

parameters = [
    RangeParameterConfig(
        name="R1",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R2",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="R3",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W1",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W2",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W3",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="D",
        bounds=bound_D,
        parameter_type="int",
    ),
]

client.configure_experiment(
    name="parn_exp",
    parameters=parameters,
)

In [26]:
MAX_SIGMA_ADVANCING = 250
MAX_SIGMA_RETREATING = 250

client.configure_optimization(
    objective="-sigma_advancing, -sigma_retreating", # TODO: double check the sign
    outcome_constraints=[
        f"sigma_advancing <= {MAX_SIGMA_ADVANCING}",
        f"sigma_retreating <= {MAX_SIGMA_RETREATING}",
        "visual_quality >= 0.5",
    ],
)

In [27]:
def cal_mean_sem(row, cols):
    filt = row[cols].notna()
    arr = row[cols][filt].astype(float).values

    if len(arr) <= 1:
        raise ValueError("Not enough data points to calculate mean and SEM.")

    mean = arr.mean()
    # NOTE: The ddof=1 because this is a sample standard deviation, not population.
    sem = arr.std(ddof=1) / np.sqrt(len(arr))
    return mean, sem

In [28]:
for idx, row in exp.iterrows():
    trial_index = client.attach_trial(

        # If Excel round-trips values as floats (e.g. 1499.9999999 due to float storage), 
        # int() truncates down instead of rounding, silently shifting your attached parameter by 1. 
        # Safer: int(round(row["R1"])).
        parameters={
            "R1": int(round(row["R1"])),
            "R2": int(round(row["R2"])),
            "R3": int(round(row["R3"])),
            "W1": int(round(row["W1"])),
            "W2": int(round(row["W2"])),
            "W3": int(round(row["W3"])),
            "D": int(round(row["D"])),
        }
    )
    advancing_mean, advancing_sem = cal_mean_sem(row, cols_advancing)
    retreating_mean, retreating_sem = cal_mean_sem(row, cols_retreating)
    visual_quality = float(row["visual_quality"])
    print(f"Trial {trial_index}: mean={advancing_mean}, sem={advancing_sem}")
    print(f"Trial {trial_index}: mean={retreating_mean}, sem={retreating_sem}")
    print(f"Trial {trial_index}: visual_quality={visual_quality}")
    client.complete_trial(
        trial_index=trial_index,
        raw_data={
            "sigma_advancing": (advancing_mean, advancing_sem),
            "sigma_retreating": (retreating_mean, retreating_sem),
            "visual_quality": visual_quality,
        },
    )

[INFO 07-31 09:45:52] ax.api.client: Trial 0 marked COMPLETED.


[INFO 07-31 09:45:52] ax.api.client: Trial 1 marked COMPLETED.
[INFO 07-31 09:45:52] ax.api.client: Trial 2 marked COMPLETED.


Trial 0: mean=205.76912250064527, sem=0.6812437108137768
Trial 0: mean=183.33003664342468, sem=0.800407713878358
Trial 0: visual_quality=1.0
Trial 1: mean=199.46360236087176, sem=0.4837853857502648
Trial 1: mean=174.82205335559163, sem=2.0185459453385746
Trial 1: visual_quality=1.0
Trial 2: mean=208.82452664375742, sem=0.6637911584527424
Trial 2: mean=188.794082691231, sem=3.9543687169767194
Trial 2: visual_quality=1.0
Trial 3: mean=212.4531422642889, sem=4.037660406660518
Trial 3: mean=190.1584666502715, sem=0.3110239767350692
Trial 3: visual_quality=1.0


[INFO 07-31 09:45:52] ax.api.client: Trial 3 marked COMPLETED.
[INFO 07-31 09:45:52] ax.api.client: Trial 4 marked COMPLETED.
[INFO 07-31 09:45:52] ax.api.client: Trial 5 marked COMPLETED.
[INFO 07-31 09:45:52] ax.api.client: Trial 6 marked COMPLETED.
[INFO 07-31 09:45:52] ax.api.client: Trial 7 marked COMPLETED.
[INFO 07-31 09:45:52] ax.api.client: Trial 8 marked COMPLETED.
[INFO 07-31 09:45:52] ax.api.client: Trial 9 marked COMPLETED.


Trial 4: mean=205.6661286329495, sem=2.117672803332696
Trial 4: mean=182.0611007893424, sem=2.0507316592325986
Trial 4: visual_quality=1.0
Trial 5: mean=198.3602960855219, sem=0.9319711479187283
Trial 5: mean=179.86334974926206, sem=0.9115969250266834
Trial 5: visual_quality=1.0
Trial 6: mean=215.58354854333197, sem=4.0172422114512445
Trial 6: mean=196.38249965923868, sem=1.353019841923621
Trial 6: visual_quality=1.0
Trial 7: mean=206.20454773775805, sem=0.4962551389455598
Trial 7: mean=182.24467956381355, sem=0.49425812993425927
Trial 7: visual_quality=0.0
Trial 8: mean=194.26050539237926, sem=0.3899675853274631
Trial 8: mean=172.83101286292012, sem=2.0471803421374326
Trial 8: visual_quality=1.0
Trial 9: mean=198.16884910201043, sem=1.6475558617556763
Trial 9: mean=179.11664695283557, sem=1.4167179487316097
Trial 9: visual_quality=0.0


In [29]:
if exp_todo.shape[0] == 0:
    suggest_params = []
    try:
        # Request only the intended number of new trials once.
        data = client.get_next_trials(max_trials=NUM_NEW_EXP)
        for trial_index, parameters in data.items():
            print(f"Trial {trial_index} with parameters {parameters}")
            suggest_params.append({"trial_index": trial_index, **parameters})
    except Exception as e:
        if type(e).__name__ == "DataRequiredError":
            print(
                "Ax cannot generate more trials yet: more observed trial data is "
                "required before moving past Sobol."
            )
        else:
            raise

    if suggest_params:
        _exp = pd.DataFrame(suggest_params)
        _exp[cols_advancing + cols_retreating] = pd.NA
        exp = pd.concat([exp, _exp], ignore_index=True).reset_index(drop=True)
        display(exp)
        SAVE_EXCEL = True
    else:
        SAVE_EXCEL = False
else:
    print("There are still experiment to be conducted. Skip generating new sets")
    SAVE_EXCEL = False

[INFO 07-31 09:45:52] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol', use_existing_trials_for_initialization=True), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 07-31 10:06:36] ax.api.client: Generated new trial 10 with parameters {'R1': 1754, 'R2': 1491, 'R3': 1813, 'W1': 47, 'W2': 70, 'W3': 81, 'D': 27} using GenerationNode Sobol.
[INFO 07-31 10:06:36] ax.api.client: Generated 

Trial 10 with parameters {'R1': 1754, 'R2': 1491, 'R3': 1813, 'W1': 47, 'W2': 70, 'W3': 81, 'D': 27}
Trial 11 with parameters {'R1': 1429, 'R2': 1667, 'R3': 1704, 'W1': 70, 'W2': 56, 'W3': 86, 'D': 25}
Trial 12 with parameters {'R1': 1454, 'R2': 1400, 'R3': 1400, 'W1': 100, 'W2': 40, 'W3': 75, 'D': 30}
Trial 13 with parameters {'R1': 1456, 'R2': 1991, 'R3': 2000, 'W1': 57, 'W2': 40, 'W3': 79, 'D': 30}
Trial 14 with parameters {'R1': 1456, 'R2': 1400, 'R3': 2000, 'W1': 100, 'W2': 92, 'W3': 85, 'D': 30}
Trial 15 with parameters {'R1': 1445, 'R2': 1578, 'R3': 2000, 'W1': 61, 'W2': 46, 'W3': 88, 'D': 10}
Trial 16 with parameters {'R1': 1446, 'R2': 2000, 'R3': 2000, 'W1': 100, 'W2': 40, 'W3': 99, 'D': 17}
Trial 17 with parameters {'R1': 1429, 'R2': 1421, 'R3': 1400, 'W1': 41, 'W2': 40, 'W3': 64, 'D': 30}
Trial 18 with parameters {'R1': 1449, 'R2': 1400, 'R3': 1457, 'W1': 100, 'W2': 40, 'W3': 79, 'D': 10}
Trial 19 with parameters {'R1': 1483, 'R2': 1400, 'R3': 2000, 'W1': 40, 'W2': 98, 'W3':

,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,1.0,205.512265,204.73876,207.056342,183.713911,181.792203,184.483996
1,1,1462,1811,1851,93,63,41,17,1.0,199.002716,200.430817,198.957274,170.864008,177.489548,176.112604
2,2,1821,1598,1421,62,74,99,20,1.0,210.138445,208.00304,208.332095,180.967275,193.690799,191.724174
3,3,1894,1873,1756,73,98,82,14,1.0,218.672547,204.882794,213.804086,189.691174,190.036535,190.747691
4,4,1691,1519,1622,51,41,57,29,1.0,202.87354,204.304761,209.820085,185.025997,178.124362,183.032944
5,5,1561,1943,1492,42,48,47,12,1.0,198.329154,196.761871,199.989863,179.458487,181.605291,178.526271
6,6,1957,1466,1959,83,90,92,26,1.0,218.466581,220.636707,207.647358,193.740708,197.195762,198.211029
7,7,1730,1704,1664,64,83,75,19,0.0,206.1824,207.074947,205.356296,181.909813,183.217576,181.606649
8,8,1489,1687,1835,92,57,66,23,1.0,193.481722,194.686593,194.613201,174.629257,168.746364,175.117417
9,9,1549,1920,1641,78,74,59,11,0.0,197.976671,201.113731,195.416145,179.738015,181.200059,176.411867


In [30]:
if SAVE_EXCEL:
    exp.to_excel(folder_path / "experiments.xlsx", index=False)

In [35]:
# import time

# t0 = time.time()
# client.get_next_trials(max_trials=1)
# print(f"{time.time()-t0:.1f}s")

# t0 = time.time()
# client.get_next_trials(max_trials=1)
# print(f"{time.time()-t0:.1f}s")

# t0 = time.time()
# client.get_next_trials(max_trials=1)
# print(f"{time.time()-t0:.1f}s")

[INFO 07-31 10:23:05] ax.api.client: Generated new trial 22 with parameters {'R1': 1422, 'R2': 1400, 'R3': 2000, 'W1': 100, 'W2': 96, 'W3': 88, 'D': 12} using GenerationNode MBM.


90.1s


[INFO 07-31 10:25:06] ax.api.client: Generated new trial 23 with parameters {'R1': 1503, 'R2': 1400, 'R3': 1400, 'W1': 40, 'W2': 46, 'W3': 100, 'D': 25} using GenerationNode MBM.


120.7s


[INFO 07-31 10:25:47] ax.api.client: Generated new trial 24 with parameters {'R1': 1429, 'R2': 1868, 'R3': 1400, 'W1': 100, 'W2': 100, 'W3': 84, 'D': 30} using GenerationNode MBM.


41.1s
